In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

## Constants

In [2]:
ENSEMBLE_MEMBERS = 600
INPUT_FILE = Path("output/102_netzero_timings.csv")
HARMONISED_DATA_FILE = "output/101_harmonised_data.csv"
NETZERO_EXTENDED_FILE = "output/102_netzero_co2_extended.csv"
NETZERO_EXTENDED_FILE_GHG = "output/102_netzero_ghg_extended.csv"
RANDOM_SEED = 42
TRIAL_SAMPLE_SIZE = 5
VERSION = "v0.1"

# Prepare Scenarios for MAGICC Runs

This notebook prepares scenario batches for running climate projections with MAGICC. 
It creates distinct batches from scenarios that achieve net-zero emissions:

- **Batch A**: Original scenarios with all forcers (ALL) - filtered for net-zero CO2
- **Batch B**: Original scenarios with CO2 only - filtered for net-zero CO2
- **Batch C**: Original scenarios with GHG only - filtered for net-zero CO2
- **Batch D**: Net-zero CO2 extended scenarios with CO2 only - filtered for net-zero CO2
- **Batch E**: Net-zero GHG extended scenarios with all forcers (ALL) - filtered for net-zero GHG

Each batch contains model-scenario combinations with different MAGICC configuration 
flags and input files.

In [3]:
df = pd.read_csv(INPUT_FILE)

## Load and Filter Data

Check the total number of scenarios in the input data.

In [4]:
n_total = len(df)
print(f"There are {n_total} scenarios")

There are 1501 scenarios


In [5]:
df

,model,scenario,year_netzero_co2,year_netzero_kyoto_ar6gwp100,achieve_netzero_co2,achieve_netzero_ghg
0,AIM/CGE 2.0,SSP1-19,2058.0,2073.0,True,True
1,AIM/CGE 2.0,SSP1-26,NaN,NaN,False,False
2,AIM/CGE 2.0,SSP1-34,NaN,NaN,False,False
3,AIM/CGE 2.0,SSP1-45,NaN,NaN,False,False
4,AIM/CGE 2.0,SSP1-Baseline,NaN,NaN,False,False
...,...,...,...,...,...,...
1496,WITCH-GLOBIOM 4.4,CD-LINKS-NPi,NaN,NaN,False,False
1497,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,2079.0,2093.0,True,True
1498,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1600,2098.0,NaN,True,False
1499,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,2059.0,2071.0,True,True


Filter for scenarios that achieve net-zero CO2 emissions and set the model-scenario index.

In [6]:
df_for_initial_batches = (
    df[df["achieve_netzero_co2"]]
    .set_index(["model", "scenario"])
)

## Create Batch Dataframes

### Batch A: Original scenarios with all forcers

In [7]:
df_batch_a = (
    df_for_initial_batches
    .copy()
    .assign(
        ENSEMBLE_MEMBERS=ENSEMBLE_MEMBERS,
        FILE=HARMONISED_DATA_FILE,
        MAGICC_FLAG="ALL"
    )
    .reset_index()
    .rename(columns={"model": "MODEL", "scenario": "SCENARIO"})
    .drop(columns=[
        "year_netzero_co2",
        "year_netzero_kyoto_ar6gwp100",
        "achieve_netzero_co2",
        "achieve_netzero_ghg"
    ])
)

### Batch B: Original scenarios with CO2 only

In [8]:
df_batch_b = df_batch_a.assign(MAGICC_FLAG="CO2")

### Batch C: Original scenarios with GHG only

In [9]:
df_batch_c = df_batch_a.assign(MAGICC_FLAG="GHG")

### Batch D: Net-zero CO2 extended scenarios with CO2 only

In [10]:
df_batch_d = df_batch_b.assign(FILE=NETZERO_EXTENDED_FILE)

In [11]:
df_batch_d

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19,600,output/102_netzero_co2_extended.csv,CO2
1,AIM/CGE 2.0,SSP2-19,600,output/102_netzero_co2_extended.csv,CO2
2,AIM/CGE 2.0,SSP2-26,600,output/102_netzero_co2_extended.csv,CO2
3,AIM/CGE 2.0,SSP3-34,600,output/102_netzero_co2_extended.csv,CO2
4,AIM/CGE 2.0,SSP4-26,600,output/102_netzero_co2_extended.csv,CO2
...,...,...,...,...,...
738,WITCH-GLOBIOM 4.4,CD-LINKS-INDC2030i_1600,600,output/102_netzero_co2_extended.csv,CO2
739,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000,600,output/102_netzero_co2_extended.csv,CO2
740,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,600,output/102_netzero_co2_extended.csv,CO2
741,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1600,600,output/102_netzero_co2_extended.csv,CO2


## Create Trial Dataframe

Sample a subset of scenarios for testing. This randomly selects scenarios from Batch D 
and creates a combined trial dataframe with the same scenarios from all four batches.

In [12]:
np.random.seed(RANDOM_SEED)
sample_indices = np.random.choice(
    df_batch_d.index,
    size=TRIAL_SAMPLE_SIZE,
    replace=False
)

trial_df = pd.concat(
    [
        df_batch_a.loc[sample_indices],
        df_batch_b.loc[sample_indices],
        df_batch_c.loc[sample_indices],
        df_batch_d.loc[sample_indices],
    ],
    ignore_index=True,
)

trial_df

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,REMIND-MAgPIE 2.0-4.1,Diff-1300Gt-hybrid-def,600,output/101_harmonised_data.csv,ALL
1,REMIND 3.0,ENGAGE-Feasibility-Maximum-Effort/Enablers & I...,600,output/101_harmonised_data.csv,ALL
2,WITCH 5.0,ENGAGE-INDCi2030-1000f-NDCp,600,output/101_harmonised_data.csv,ALL
3,MESSAGEix-GLOBIOM 1.1,ENGAGE-NPi2020-1000,600,output/101_harmonised_data.csv,ALL
4,IMAGE 3.0.1,EMF30-ClimatePolicy+SLCF,600,output/101_harmonised_data.csv,ALL
5,REMIND-MAgPIE 2.0-4.1,Diff-1300Gt-hybrid-def,600,output/101_harmonised_data.csv,CO2
6,REMIND 3.0,ENGAGE-Feasibility-Maximum-Effort/Enablers & I...,600,output/101_harmonised_data.csv,CO2
7,WITCH 5.0,ENGAGE-INDCi2030-1000f-NDCp,600,output/101_harmonised_data.csv,CO2
8,MESSAGEix-GLOBIOM 1.1,ENGAGE-NPi2020-1000,600,output/101_harmonised_data.csv,CO2
9,IMAGE 3.0.1,EMF30-ClimatePolicy+SLCF,600,output/101_harmonised_data.csv,CO2


### Batch E: Net-zero GHG extended scenarios with all forcers

This batch uses a different filter (achieve_netzero_ghg) and the GHG extended file.

In [13]:
df_batch_e = (
    df[df["achieve_netzero_ghg"]]
    .copy()
    .assign(
        ENSEMBLE_MEMBERS=ENSEMBLE_MEMBERS,
        FILE=NETZERO_EXTENDED_FILE_GHG,
        MAGICC_FLAG="ALL"
    )
    .rename(columns={"model": "MODEL", "scenario": "SCENARIO"})
    .drop(columns=[
        "year_netzero_co2",
        "year_netzero_kyoto_ar6gwp100",
        "achieve_netzero_co2",
        "achieve_netzero_ghg"
    ])
)

df_batch_e

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19,600,output/102_netzero_ghg_extended.csv,ALL
5,AIM/CGE 2.0,SSP2-19,600,output/102_netzero_ghg_extended.csv,ALL
29,AIM/CGE 2.1,CD-LINKS-NPi2020_400,600,output/102_netzero_ghg_extended.csv,ALL
34,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional,600,output/102_netzero_ghg_extended.csv,ALL
40,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutional,600,output/102_netzero_ghg_extended.csv,ALL
...,...,...,...,...,...
1460,WITCH-GLOBIOM 3.1,SSP1-19,600,output/102_netzero_ghg_extended.csv,ALL
1474,WITCH-GLOBIOM 3.1,SSP4-19,600,output/102_netzero_ghg_extended.csv,ALL
1495,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000,600,output/102_netzero_ghg_extended.csv,ALL
1497,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,600,output/102_netzero_ghg_extended.csv,ALL


## Concatenate All Batches

In [14]:
df_all_batches = pd.concat(
    [df_batch_a, df_batch_b, df_batch_c, df_batch_d, df_batch_e],
    ignore_index=True,
)

df_all_batches

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19,600,output/101_harmonised_data.csv,ALL
1,AIM/CGE 2.0,SSP2-19,600,output/101_harmonised_data.csv,ALL
2,AIM/CGE 2.0,SSP2-26,600,output/101_harmonised_data.csv,ALL
3,AIM/CGE 2.0,SSP3-34,600,output/101_harmonised_data.csv,ALL
4,AIM/CGE 2.0,SSP4-26,600,output/101_harmonised_data.csv,ALL
...,...,...,...,...,...
3295,WITCH-GLOBIOM 3.1,SSP1-19,600,output/102_netzero_ghg_extended.csv,ALL
3296,WITCH-GLOBIOM 3.1,SSP4-19,600,output/102_netzero_ghg_extended.csv,ALL
3297,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000,600,output/102_netzero_ghg_extended.csv,ALL
3298,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,600,output/102_netzero_ghg_extended.csv,ALL


In [15]:
df_all_batches.to_csv(
    Path(
        f"output/300_all_batches_{VERSION}.csv"
    )
)

In [16]:
trial_df.to_csv(
    Path(
        f"output/300_trial_batches_{VERSION}.csv"
    )
)